# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [18]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

print("Libraries loaded.")

Libraries loaded.


In [9]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

PRE_MONTHS = "('2025-07','2025-08','2025-09')"
A_MONTHS = "('2025-10','2025-11','2025-12')"
B_MONTHS = "('2026-01','2026-02','2026-03')"

print("Warehouse connected.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Warehouse connected.


In [10]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "month",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Required warehouse columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required warehouse columns are available.


In [11]:
pre = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_pre
    FROM warehouse
    WHERE month IN {PRE_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

feat_a = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS total_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_a,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position,
        ROUND(STDDEV(gsc_avg_position), 2) AS position_volatility,
        ROUND(
            SUM(COALESCE(gsc_clicks, 0)) * 1.0
            / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0),
            4
        ) AS ctr,
        COUNT(
            DISTINCT CASE
                WHEN gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS active_gsc_days
    FROM warehouse
    WHERE month IN {A_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

label_b = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_b
    FROM warehouse
    WHERE month IN {B_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

df = feat_a.merge(
    label_b,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

df = df.merge(
    pre,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

df = df[df["total_impressions"] > 0].copy()

print("Merged eligible pairs:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged eligible pairs: 110933


,client_hash_id,content_hash_id,total_impressions,clicks_a,avg_position,position_volatility,ctr,active_gsc_days,clicks_b,clicks_pre
0,client_fef1a8f436438636,content_27d8221f9191e369,960.0,0.0,6.54,6.17,0.0000,92,0.0,0.0
1,client_fef1a8f436438636,content_064b174854b62a19,9178.0,3.0,7.36,1.22,0.0003,92,6.0,1.0
2,client_fef1a8f436438636,content_9732385705d7914d,2815.0,8.0,3.66,2.89,0.0028,92,21.0,0.0
3,client_fef1a8f436438636,content_44677b2ffd7549a9,5437.0,35.0,3.40,1.72,0.0064,83,60.0,5.0
4,client_fef1a8f436438636,content_84b4a576a4379876,3324.0,4.0,2.81,1.64,0.0012,92,13.0,0.0


In [12]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "month",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Required warehouse columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required warehouse columns are available.


In [13]:
pre = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_pre
    FROM warehouse
    WHERE month IN {PRE_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

feat_a = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS total_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_a,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position,
        ROUND(STDDEV(gsc_avg_position), 2) AS position_volatility,
        ROUND(
            SUM(COALESCE(gsc_clicks, 0)) * 1.0
            / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0),
            4
        ) AS ctr,
        COUNT(
            DISTINCT CASE
                WHEN gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS active_gsc_days
    FROM warehouse
    WHERE month IN {A_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

label_b = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_b
    FROM warehouse
    WHERE month IN {B_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

df = feat_a.merge(
    label_b,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

df = df.merge(
    pre,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

df = df[df["total_impressions"] > 0].copy()

print("Merged eligible pairs:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged eligible pairs: 110933


,client_hash_id,content_hash_id,total_impressions,clicks_a,avg_position,position_volatility,ctr,active_gsc_days,clicks_b,clicks_pre
0,client_fef1a8f436438636,content_27d8221f9191e369,960.0,0.0,6.54,6.17,0.0000,92,0.0,0.0
1,client_fef1a8f436438636,content_064b174854b62a19,9178.0,3.0,7.36,1.22,0.0003,92,6.0,1.0
2,client_fef1a8f436438636,content_9732385705d7914d,2815.0,8.0,3.66,2.89,0.0028,92,21.0,0.0
3,client_fef1a8f436438636,content_44677b2ffd7549a9,5437.0,35.0,3.40,1.72,0.0064,83,60.0,5.0
4,client_fef1a8f436438636,content_84b4a576a4379876,3324.0,4.0,2.81,1.64,0.0012,92,13.0,0.0


In [14]:
df["pct_change"] = (
    (df["clicks_b"] - df["clicks_a"])
    / df["clicks_a"].replace(0, np.nan)
)

df["pct_change_pre_to_a"] = (
    (df["clicks_a"] - df["clicks_pre"])
    / df["clicks_pre"].replace(0, np.nan)
)

def label_row(row):
    if pd.isna(row["pct_change"]):
        return "worth_review"

    was_declining = (
        pd.notna(row["pct_change_pre_to_a"])
        and row["pct_change_pre_to_a"] < -0.10
    )

    if was_declining and row["pct_change"] > 0.05:
        return "recovering"

    if row["pct_change"] > 0.20:
        return "growing"

    if row["pct_change"] < -0.20:
        return "declining"

    return "worth_review"

df["label"] = df.apply(label_row, axis=1)

df["volume_tier"] = pd.qcut(
    df["total_impressions"],
    q=3,
    labels=["low", "med", "high"],
    duplicates="drop",
)

FEATURES = [
    "avg_position",
    "position_volatility",
    "ctr",
    "total_impressions",
    "active_gsc_days",
]

df_model = df.dropna(
    subset=FEATURES + ["label", "volume_tier"]
).copy()

X = pd.get_dummies(
    df_model[FEATURES + ["volume_tier"]],
    columns=["volume_tier"],
    dtype=int,
)

y = df_model["label"].copy()

pair_groups = (
    df_model["client_hash_id"].astype(str)
    + "_"
    + df_model["content_hash_id"].astype(str)
)

client_groups = df_model["client_hash_id"].astype(str)

print("Feature matrix shape:", X.shape)
print("Label distribution:")
display(y.value_counts().to_frame("n"))
display(y.value_counts(normalize=True).round(4).to_frame("rate"))

Feature matrix shape: (103933, 8)
Label distribution:


,n
label,
worth_review,62757
growing,23186
declining,16687
recovering,1303


,rate
label,
worth_review,0.6038
growing,0.2231
declining,0.1606
recovering,0.0125


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Paper finding 1

The capstone reports **0.802 model accuracy** against a **0.600 majority-class baseline**, a measured lift of **+0.202**, using a Random Forest and a 75/25 grouped split.

**Methodology question:** Does the lift remain when the test set contains entirely unseen clients rather than unseen `(client, content)` pairs from clients that may also appear in training?

**Why this matters:** The capstone has one aggregated row per pair. Grouping by pair prevents the same pair appearing twice, but it does not prevent pages belonging to the same client from appearing on both sides. Client-level practices may therefore transfer across the original split.

### Paper finding 2

The capstone reports strong performance for `worth_review` and useful recall for `growing`, but extremely weak recall for `recovering`.

**Methodology question:** Is the poor `recovering` performance mainly caused by class imbalance, or is the label itself too difficult to infer from Window A features without explicit pre-window trend features?

**Why this matters:** The pre-window information helps define `recovering` but is intentionally excluded from the feature matrix. This protects against label leakage, but it may also make recovery difficult to identify.

These questions audit the capstone's methodology without rejecting its observed results.

In [15]:
findings_audit = pd.DataFrame([
    {
        "paper_finding": (
            "Accuracy 0.802 versus majority baseline 0.600; lift +0.202."
        ),
        "methodology_question": (
            "Does the lift remain under a fully unseen-client split?"
        ),
        "risk_being_tested": (
            "Client-level patterns may appear in both train and test "
            "under pair-level grouping."
        ),
    },
    {
        "paper_finding": (
            "worth_review performs strongly; recovering recall is very weak."
        ),
        "methodology_question": (
            "Is recovering weak because of rarity, label design, "
            "or missing pre-window trend information?"
        ),
        "risk_being_tested": (
            "The model may lack enough examples or enough safe features "
            "to distinguish recovery."
        ),
    },
])

display(findings_audit)

,paper_finding,methodology_question,risk_being_tested
0,Accuracy 0.802 versus majority baseline 0.600; lift +0.202.,Does the lift remain under a fully unseen-client split?,Client-level patterns may appear in both train and test under pair-level grouping.
1,worth_review performs strongly; recovering recall is very weak.,"Is recovering weak because of rarity, label design, or missing pre-window trend information?",The model may lack enough examples or enough safe features to distinguish recovery.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compare two validation designs using the same Random Forest, feature matrix, labels, and 75/25 ratio:

1. **Capstone split:** grouped by `(client_hash_id, content_hash_id)`.
2. **Stricter audit split:** grouped by `client_hash_id`.

Because the modeling frame already has one row per pair, pair-level grouping behaves similarly to a row-level split. The client-level split is stricter because no client can appear in both train and test data.

The client-level result is treated as the primary audit result.

In [19]:
CLASS_ORDER = [
    "declining",
    "growing",
    "recovering",
    "worth_review",
]

def fit_and_evaluate(split_name, train_index, test_index):
    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()
    y_train = y.iloc[train_index].copy()
    y_test = y.iloc[test_index].copy()

    train_clients = set(
        df_model.iloc[train_index]["client_hash_id"]
    )
    test_clients = set(
        df_model.iloc[test_index]["client_hash_id"]
    )

    majority_class = y_train.value_counts().idxmax()
    baseline_predictions = np.repeat(
        majority_class,
        len(y_test),
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics = {
        "split": split_name,
        "train_rows": int(len(train_index)),
        "test_rows": int(len(test_index)),
        "train_clients": int(len(train_clients)),
        "test_clients": int(len(test_clients)),
        "client_overlap": int(
            len(train_clients.intersection(test_clients))
        ),
        "majority_class": majority_class,
        "baseline_accuracy": float(
            accuracy_score(y_test, baseline_predictions)
        ),
        "model_accuracy": float(
            accuracy_score(y_test, predictions)
        ),
        "accuracy_lift": float(
            accuracy_score(y_test, predictions)
            - accuracy_score(y_test, baseline_predictions)
        ),
        "macro_f1": float(
            f1_score(
                y_test,
                predictions,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_test,
                predictions,
                labels=CLASS_ORDER,
                average="weighted",
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_test, predictions)
        ),
    }

    report = pd.DataFrame(
        classification_report(
            y_test,
            predictions,
            labels=CLASS_ORDER,
            output_dict=True,
            zero_division=0,
        )
    ).T

    prediction_frame = df_model.iloc[test_index][
        [
            "client_hash_id",
            "content_hash_id",
            "total_impressions",
            "clicks_a",
            "clicks_b",
            "avg_position",
            "position_volatility",
            "ctr",
            "active_gsc_days",
            "label",
        ]
    ].copy()

    prediction_frame["predicted_label"] = predictions
    prediction_frame["correct"] = (
        prediction_frame["label"]
        == prediction_frame["predicted_label"]
    )

    return metrics, report, model, prediction_frame

In [20]:
pair_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

pair_train_index, pair_test_index = next(
    pair_splitter.split(
        X,
        y,
        groups=pair_groups,
    )
)

client_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

client_train_index, client_test_index = next(
    client_splitter.split(
        X,
        y,
        groups=client_groups,
    )
)

(
    pair_metrics,
    pair_report,
    pair_model,
    pair_predictions,
) = fit_and_evaluate(
    "capstone_pair_group_split",
    pair_train_index,
    pair_test_index,
)

(
    client_metrics,
    client_report,
    client_model,
    client_predictions,
) = fit_and_evaluate(
    "strict_unseen_client_split",
    client_train_index,
    client_test_index,
)

comparison = pd.DataFrame(
    [pair_metrics, client_metrics]
)

display(comparison.round(4))

,split,train_rows,test_rows,train_clients,test_clients,client_overlap,majority_class,baseline_accuracy,model_accuracy,accuracy_lift,macro_f1,weighted_f1,balanced_accuracy
0,capstone_pair_group_split,77949,25984,36,34,34,worth_review,0.5999,0.8005,0.2005,0.5502,0.8038,0.5619
1,strict_unseen_client_split,93327,10606,27,9,0,worth_review,0.7507,0.8105,0.0598,0.4437,0.8254,0.4893


In [24]:
print("Capstone pair-group split per-class report")
display(
    pair_report.loc[
        CLASS_ORDER,
        ["precision", "recall", "f1-score", "support"],
    ].round(4)
)

print("Strict unseen-client split per-class report")
display(
    client_report.loc[
        CLASS_ORDER,
        ["precision", "recall", "f1-score", "support"],
    ].round(4)
)

Capstone pair-group split per-class report


,precision,recall,f1-score,support
declining,0.5564,0.5825,0.5691,4177.0
growing,0.6040,0.7584,0.6724,5864.0
recovering,0.1351,0.0141,0.0256,354.0
worth_review,0.9791,0.8926,0.9338,15589.0


Strict unseen-client split per-class report


,precision,recall,f1-score,support
declining,0.5936,0.3436,0.4352,1790.0
growing,0.2595,0.6796,0.3756,802.0
recovering,0.0000,0.0000,0.0000,52.0
worth_review,0.9956,0.9339,0.9638,7962.0


In [25]:
pair_train_clients = set(
    df_model.iloc[pair_train_index]["client_hash_id"]
)
pair_test_clients = set(
    df_model.iloc[pair_test_index]["client_hash_id"]
)

client_train_clients = set(
    df_model.iloc[client_train_index]["client_hash_id"]
)
client_test_clients = set(
    df_model.iloc[client_test_index]["client_hash_id"]
)

pair_client_overlap = len(
    pair_train_clients.intersection(pair_test_clients)
)

strict_client_overlap = len(
    client_train_clients.intersection(client_test_clients)
)

print("Pair-group split client overlap:", pair_client_overlap)
print("Unseen-client split client overlap:", strict_client_overlap)

assert strict_client_overlap == 0

accuracy_change = (
    client_metrics["model_accuracy"]
    - pair_metrics["model_accuracy"]
)

macro_f1_change = (
    client_metrics["macro_f1"]
    - pair_metrics["macro_f1"]
)

print("Unseen-client minus pair-split accuracy:", round(accuracy_change, 4))
print("Unseen-client minus pair-split macro F1:", round(macro_f1_change, 4))

Pair-group split client overlap: 34
Unseen-client split client overlap: 0
Unseen-client minus pair-split accuracy: 0.01
Unseen-client minus pair-split macro F1: -0.1066


In [26]:
print("Strict unseen-client confusion matrix")

client_confusion = confusion_matrix(
    client_predictions["label"],
    client_predictions["predicted_label"],
    labels=CLASS_ORDER,
)

display(
    pd.DataFrame(
        client_confusion,
        index=[f"actual_{label}" for label in CLASS_ORDER],
        columns=[f"predicted_{label}" for label in CLASS_ORDER],
    )
)

error_summary = (
    client_predictions.loc[
        ~client_predictions["correct"],
        ["label", "predicted_label"],
    ]
    .value_counts()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

display(error_summary.head(15))

Strict unseen-client confusion matrix


,predicted_declining,predicted_growing,predicted_recovering,predicted_worth_review
actual_declining,615,1157,0,18
actual_growing,242,545,1,14
actual_recovering,18,33,0,1
actual_worth_review,161,365,0,7436


,label,predicted_label,n
0,declining,growing,1157
1,worth_review,growing,365
2,growing,declining,242
3,worth_review,declining,161
4,recovering,growing,33
5,declining,worth_review,18
6,recovering,declining,18
7,growing,worth_review,14
8,growing,recovering,1
9,recovering,worth_review,1


### Validation interpretation

The pair-group result reproduces the capstone's evaluation design. The unseen-client split asks a harder deployment question: whether the learned rules transfer to clients absent from training.

If performance falls, the original result was partly dependent on client-specific patterns. If it remains similar, that strengthens the case for transferability. Either outcome is an observed result from one split, not proof of universal generalization.

`recovering` should remain a low-confidence class unless recall improves materially under repeated validation.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature matrix is checked against the capstone's own leakage rules.

Allowed model inputs:

- `avg_position`
- `position_volatility`
- `ctr`
- `total_impressions`
- `active_gsc_days`
- one-hot encoded `volume_tier`

Explicitly excluded:

- Window B clicks
- Percentage-change columns
- The final label
- Pre-computed trend fields
- Client/content identifiers
- Client names, domains, URLs, and queries

In [21]:
forbidden_terms = [
    "clicks_b",
    "clicks_pre",
    "pct_change",
    "label",
    "target",
    "future",
    "trend",
    "client_hash_id",
    "content_hash_id",
    "client_name",
    "company_name",
    "domain",
    "url",
    "query",
    "keyword",
    "page_title",
    "product_flag",
    "refresh_flag",
]

leakage_hits = [
    column
    for column in X.columns
    if any(
        term in column.lower()
        for term in forbidden_terms
    )
]

expected_feature_columns = {
    "avg_position",
    "position_volatility",
    "ctr",
    "total_impressions",
    "active_gsc_days",
    "volume_tier_low",
    "volume_tier_med",
    "volume_tier_high",
}

unexpected_features = [
    column
    for column in X.columns
    if column not in expected_feature_columns
]

missing_expected_features = [
    column
    for column in expected_feature_columns
    if column not in X.columns
]

print("Feature columns:")
for column in X.columns:
    print("-", column)

print("\nLeakage hits:", leakage_hits)
print("Unexpected features:", unexpected_features)
print("Missing expected features:", missing_expected_features)

assert leakage_hits == []
assert unexpected_features == []
assert missing_expected_features == []

print("PASS: The model matrix contains only the approved Window A features.")

Feature columns:
- avg_position
- position_volatility
- ctr
- total_impressions
- active_gsc_days
- volume_tier_low
- volume_tier_med
- volume_tier_high

Leakage hits: []
Unexpected features: []
Missing expected features: []
PASS: The model matrix contains only the approved Window A features.


In [22]:
window_a_check = con.execute(f'''
    SELECT
        MIN(report_date) AS minimum_feature_date,
        MAX(report_date) AS maximum_feature_date,
        COUNT(*) AS source_rows
    FROM warehouse
    WHERE month IN {A_MONTHS}
      AND gsc_data_available IS TRUE
''').df()

window_b_check = con.execute(f'''
    SELECT
        MIN(report_date) AS minimum_label_date,
        MAX(report_date) AS maximum_label_date,
        COUNT(*) AS source_rows
    FROM warehouse
    WHERE month IN {B_MONTHS}
      AND gsc_data_available IS TRUE
''').df()

display(window_a_check)
display(window_b_check)

maximum_feature_date = pd.to_datetime(
    window_a_check.loc[0, "maximum_feature_date"]
)

minimum_label_date = pd.to_datetime(
    window_b_check.loc[0, "minimum_label_date"]
)

assert maximum_feature_date < minimum_label_date

print("PASS: The feature window ends before the label window begins.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,minimum_feature_date,maximum_feature_date,source_rows
0,2025-10-01,2025-12-31,4956697


,minimum_label_date,maximum_label_date,source_rows
0,2026-01-01,2026-03-31,8629987


PASS: The feature window ends before the label window begins.


In [27]:
leakage_audit = pd.DataFrame([
    {
        "check": "Future or label-derived columns in X",
        "result": "PASS" if not leakage_hits else "FAIL",
        "details": leakage_hits,
    },
    {
        "check": "Unexpected model features",
        "result": "PASS" if not unexpected_features else "FAIL",
        "details": unexpected_features,
    },
    {
        "check": "Required capstone features present",
        "result": "PASS" if not missing_expected_features else "FAIL",
        "details": missing_expected_features,
    },
    {
        "check": "Feature window precedes label window",
        "result": (
            "PASS"
            if maximum_feature_date < minimum_label_date
            else "FAIL"
        ),
        "details": (
            f"{maximum_feature_date.date()} < "
            f"{minimum_label_date.date()}"
        ),
    },
    {
        "check": "Unseen-client split has zero client overlap",
        "result": "PASS" if strict_client_overlap == 0 else "FAIL",
        "details": strict_client_overlap,
    },
])

display(leakage_audit)

,check,result,details
0,Future or label-derived columns in X,PASS,[]
1,Unexpected model features,PASS,[]
2,Required capstone features present,PASS,[]
3,Feature window precedes label window,PASS,2025-12-31 < 2026-01-01
4,Unseen-client split has zero client overlap,PASS,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Overstated claim

> The Random Forest accurately identifies which pages need refreshing and tells editors which changes will improve traffic.

### Safe claim

> On the capstone's July 2025–March 2026 warehouse sample, a Random Forest using October–December GSC features showed measured ability to classify later January–March click trajectories into `growing`, `declining`, `recovering`, and `worth_review`. Performance should be reported against the majority baseline and under both pair-group and unseen-client validation. The output is directional decision support for prioritizing human review; it does not establish that a page requires a refresh, that a suggested edit will increase traffic, or that performance will generalize to future periods.

This version states the population, windows, model, measured outcome, comparison, and causal limitations.

In [28]:
claim_audit = pd.DataFrame([
    {
        "claim_type": "overstated",
        "claim": (
            "The model identifies pages that need refreshing "
            "and tells editors which changes will improve traffic."
        ),
        "problem": (
            "It converts an observational trajectory classification "
            "into a causal intervention claim."
        ),
    },
    {
        "claim_type": "safe",
        "claim": (
            "The model provides measured, directional classification "
            "of later click trajectories and supports prioritization "
            "for human review."
        ),
        "problem": (
            "No causal effect or universal generalization is claimed."
        ),
    },
])

display(claim_audit)

,claim_type,claim,problem
0,overstated,The model identifies pages that need refreshing and tells editors which changes will improve traffic.,It converts an observational trajectory classification into a causal intervention claim.
1,safe,"The model provides measured, directional classification of later click trajectories and supports prioritization for human review.",No causal effect or universal generalization is claimed.


In [29]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUTPUT_DIR / "w06_validation_audit_metrics.json"

metrics_receipt = {
    "assignment": "ML-09 validation and research claim audit",
    "source": "Content Opportunity Scoring capstone",
    "pre_window": "2025-07 to 2025-09",
    "feature_window": "2025-10 to 2025-12",
    "label_window": "2026-01 to 2026-03",
    "model": "RandomForestClassifier",
    "classes": CLASS_ORDER,
    "capstone_pair_group_split": pair_metrics,
    "strict_unseen_client_split": client_metrics,
    "unseen_client_minus_pair_accuracy": float(accuracy_change),
    "unseen_client_minus_pair_macro_f1": float(macro_f1_change),
    "primary_audit_split": "strict_unseen_client_split",
    "leakage_hits": leakage_hits,
    "unexpected_features": unexpected_features,
    "missing_expected_features": missing_expected_features,
    "future_inputs_used_as_features": False,
    "identifiers_used_as_features": False,
    "private_fields_used": False,
}

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(
        metrics_receipt,
        file,
        indent=2,
        default=str,
    )

print("Metrics receipt written to:", metrics_path)

Metrics receipt written to: work/outputs/w06_validation_audit_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [30]:
assert strict_client_overlap == 0
assert leakage_hits == []
assert unexpected_features == []
assert missing_expected_features == []
assert maximum_feature_date < minimum_label_date
assert metrics_path.exists()

print("ML-09 COMPLETE")
print("Notebook: work/notebooks/w06_validation_audit.ipynb")
print("Metrics receipt:", metrics_path)

ML-09 COMPLETE
Notebook: work/notebooks/w06_validation_audit.ipynb
Metrics receipt: work/outputs/w06_validation_audit_metrics.json
